In [ ]:
import torch
import torch.nn as nn
import torch.utils.data as Data
import torch.nn.functional as F
import numpy as np
import tensorflow as tf

learning_rate = 1e-4
keep_prob_rate = 0.3  # Dropout probability
max_epoch = 3
BATCH_SIZE = 50

# Use Keras MNIST to avoid torchvision dependency issues
(x_train, y_train), (x_test_all, y_test_all) = tf.keras.datasets.mnist.load_data()
x_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1) / 255.0
y_train = torch.tensor(y_train, dtype=torch.long)

train_dataset = Data.TensorDataset(x_train, y_train)
train_loader = Data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_x = torch.tensor(x_test_all[:500], dtype=torch.float32).unsqueeze(1) / 255.0
test_y = y_test_all[:500]


class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                # patch 7 * 7 ; 1 in channels ; 32 out channels ; stride is 1
                # padding style is same (input and output have same size)
                in_channels=1,
                out_channels=32,
                kernel_size=7,
                stride=1,
                padding=3,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            # line 1 : convolution function, patch 5*5 , 32 in channels ;64 out channels
            # line 2 : activation function
            # line 3 : pooling operation function
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out1 = nn.Linear(7 * 7 * 64, 1024, bias=True)
        self.dropout = nn.Dropout(keep_prob_rate)
        self.out2 = nn.Linear(1024, 10, bias=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # flatten to (batch_size, 7*7*64)
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        logits = self.out2(out1)
        return logits


def test(cnn):
    with torch.no_grad():
        logits = cnn(test_x)
        prediction = torch.argmax(logits, dim=1).cpu().numpy()
    correct = np.sum(prediction == test_y)
    return correct / 500.0


def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
    loss_func = nn.CrossEntropyLoss()
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            logits = cnn(x_)
            loss = loss_func(logits, y_)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if step != 0 and step % 100 == 0:
                print("epoch", epoch, "step", step, "test accuracy", test(cnn))


if __name__ == '__main__':
    cnn = CNN()
    train(cnn)
    print("final test accuracy:", test(cnn))